### Imports

In [1]:
import pandas as pd

### Load raw data

In [2]:
df = pd.read_csv("../data/raw/raw_walks.csv")

### Explore initial data
- Shape
- Basic info
- Basic statistics
- Check for null values

There are some null values in the elevation, temperature, and humidity, but for the use case of this project that is okay

In [3]:
# Check the shape of the dataset
df.shape

(88, 18)

In [4]:
# Basic info about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   workoutActivityType  88 non-null     object 
 1   duration             88 non-null     float64
 2   durationUnit         88 non-null     object 
 3   sourceName           88 non-null     object 
 4   sourceVersion        88 non-null     object 
 5   device               88 non-null     object 
 6   creationDate         88 non-null     object 
 7   startDate            88 non-null     object 
 8   endDate              88 non-null     object 
 9   elevation_ascended   62 non-null     object 
 10  humidity             82 non-null     object 
 11  temperature          82 non-null     object 
 12  distance_km          88 non-null     float64
 13  basal_calories       88 non-null     float64
 14  heart_rate_average   88 non-null     float64
 15  heart_rate_min       88 non-null     int64

In [5]:
# Basic statistics for each column
df.describe()

,duration,distance_km,basal_calories,heart_rate_average,heart_rate_min,heart_rate_max,active_calories
count,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000
mean,36.325199,2.540119,59.896178,121.748400,99.147727,143.318182,178.005088
std,27.177739,1.919275,45.094237,13.406075,12.293854,16.660605,147.143630
min,1.101888,0.053492,0.946222,98.575000,74.000000,102.000000,2.482000
25%,19.354378,1.436992,31.891900,111.309750,90.750000,130.000000,91.588100
50%,28.950750,2.019325,47.480450,121.145500,98.000000,142.500000,137.459000
75%,44.925303,3.023022,73.831750,130.907000,109.000000,155.250000,187.903500
max,139.973238,11.585600,232.059000,150.684000,128.000000,180.000000,816.914000


In [6]:
# Check for null values in each column
df.isnull().sum()

workoutActivityType     0
duration                0
durationUnit            0
sourceName              0
sourceVersion           0
device                  0
creationDate            0
startDate               0
endDate                 0
elevation_ascended     26
humidity                6
temperature             6
distance_km             0
basal_calories          0
heart_rate_average      0
heart_rate_min          0
heart_rate_max          0
active_calories         0
dtype: int64

### Rename columns
Rename some columns for better understanding and organization

In [7]:
# View list of initial columns
df.columns

Index(['workoutActivityType', 'duration', 'durationUnit', 'sourceName',
       'sourceVersion', 'device', 'creationDate', 'startDate', 'endDate',
       'elevation_ascended', 'humidity', 'temperature', 'distance_km',
       'basal_calories', 'heart_rate_average', 'heart_rate_min',
       'heart_rate_max', 'active_calories'],
      dtype='object')

In [8]:
# Rename columns for clarity
df = df.rename(columns={
    "elevation_ascended": "elevation_ascended_cm",
    "basal_calories": "calories_basal",
    "active_calories": "calories_active",
})

### Create new columns
Create new columns that were not included in the orginal data export. 
- Pace and speed
- Ranges and totals
- Further date information

In [ ]:
# Speed and pace
df["pace"] = df["duration"] / df["distance_km"]
df["speed"] = df["distance_km"] / (df["duration"] / 60.0)

# Range and totals
df["heart_rate_range"] = df["heart_rate_max"] - df["heart_rate_min"]
df["calories_total"] = df["calories_active"] + df["calories_basal"]
df["calories_per_km"] = df["calories_total"] / (df["distance_km"])

# Date information
df["year"] = pd.to_datetime(df["startDate"]).dt.year
df["month"] = pd.to_datetime(df["startDate"]).dt.month
df["day"] = pd.to_datetime(df["startDate"]).dt.day
df["day_of_week"] = pd.to_datetime(df["startDate"]).dt.dayofweek

# Season
def season(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    return "Fall"
df["season"] = df["month"].apply(season)

# Check new list of columns
df.columns

Index(['workoutActivityType', 'duration', 'durationUnit', 'sourceName',
       'sourceVersion', 'device', 'creationDate', 'startDate', 'endDate',
       'elevation_ascended_cm', 'humidity', 'temperature', 'distance_km',
       'calories_basal', 'heart_rate_average', 'heart_rate_min',
       'heart_rate_max', 'calories_active', 'pace', 'speed',
       'heart_rate_range', 'calories_total', 'calories_per_km', 'year',
       'month', 'day', 'day_of_week', 'season'],
      dtype='object')

### Drop unnecessary columns

In [10]:
# df = df.drop(columns=["startDate", "endDate", "sourceName", "sourceVersion", "device", "workoutActivityType"])

df = df.drop(columns=["workoutActivityType", "durationUnit", "sourceName", "sourceVersion", "device", "creationDate"], errors="ignore")

df.head(2)

,duration,startDate,endDate,elevation_ascended_cm,humidity,temperature,distance_km,calories_basal,heart_rate_average,heart_rate_min,...,pace,speed,heart_rate_range,calories_total,calories_per_km,year,month,day,day_of_week,season
0,23.467609,2024-03-09 17:03:18 -0300,2024-03-09 17:26:46 -0300,1565 cm,5500 %,43.142 degF,1.70326,38.2454,106.047,82,...,13.778054,4.354751,42,122.1538,71.717647,2024,3,9,5,Spring
1,45.893084,2024-03-16 14:36:43 -0300,2024-03-16 15:23:49 -0300,5159 cm,5100 %,47.642 degF,3.02969,76.0570,106.438,74,...,15.147782,3.960976,50,263.3920,86.936947,2024,3,16,5,Spring


### Convert column types
Converting columns to proper types

In [11]:
# Check types before conversion
print(df.dtypes)

duration                 float64
startDate                 object
endDate                   object
elevation_ascended_cm     object
humidity                  object
temperature               object
distance_km              float64
calories_basal           float64
heart_rate_average       float64
heart_rate_min             int64
heart_rate_max             int64
calories_active          float64
pace                     float64
speed                    float64
heart_rate_range           int64
calories_total           float64
calories_per_km          float64
year                       int32
month                      int32
day                        int32
day_of_week                int32
season                    object
dtype: object


In [12]:
# Convert types
df["startDate"] = pd.to_datetime(df["startDate"], errors="coerce")
df["endDate"] = pd.to_datetime(df["endDate"], errors="coerce")

df["elevation_ascended_cm"] = pd.to_numeric(
    df["elevation_ascended_cm"].str.replace(r"[^\d\.-]+", "", regex=True),
    errors="coerce",
)
df["humidity"] = pd.to_numeric(
    df["humidity"].str.replace(r"[^\d\.-]+", "", regex=True),
    errors="coerce",
)
df["temperature"] = pd.to_numeric(
    df["temperature"].str.replace(r"[^\d\.-]+", "", regex=True),
    errors="coerce",
)

# Check types after conversion
print(df.dtypes)

duration                                   float64
startDate                datetime64[ns, UTC-03:00]
endDate                  datetime64[ns, UTC-03:00]
elevation_ascended_cm                      float64
humidity                                   float64
temperature                                float64
distance_km                                float64
calories_basal                             float64
heart_rate_average                         float64
heart_rate_min                               int64
heart_rate_max                               int64
calories_active                            float64
pace                                       float64
speed                                      float64
heart_rate_range                             int64
calories_total                             float64
calories_per_km                            float64
year                                         int32
month                                        int32
day                            

In [13]:
# Convert F to C
df["temperature"] = (df["temperature"] - 32) * (5.0/9.0)

df.head(2)

,duration,startDate,endDate,elevation_ascended_cm,humidity,temperature,distance_km,calories_basal,heart_rate_average,heart_rate_min,...,pace,speed,heart_rate_range,calories_total,calories_per_km,year,month,day,day_of_week,season
0,23.467609,2024-03-09 17:03:18-03:00,2024-03-09 17:26:46-03:00,1565.0,5500.0,6.19,1.70326,38.2454,106.047,82,...,13.778054,4.354751,42,122.1538,71.717647,2024,3,9,5,Spring
1,45.893084,2024-03-16 14:36:43-03:00,2024-03-16 15:23:49-03:00,5159.0,5100.0,8.69,3.02969,76.0570,106.438,74,...,15.147782,3.960976,50,263.3920,86.936947,2024,3,16,5,Spring


### Drop outliner walks
Drop walks that aren't long enough to be considered, and pose risk to skewing the data 

In [14]:
# Drop walks under half KM, and walks under 10 mins
df = df[df["distance_km"] >= 0.5]
df = df[df["duration"] >= 10]

df.shape

(84, 22)

### Drop duplicates

In [15]:
# Check for duplicates and drop
df = df.drop_duplicates()

df.shape

(84, 22)

### Reorder columns

In [16]:
# Reorder columns

newOrder = [
    "startDate",
	"endDate",
	"distance_km",
	"elevation_ascended_cm",
	"duration",
	"pace",
	"speed",
	"calories_active",
	"calories_basal",
	"calories_total",
	"calories_per_km",
	"heart_rate_average",
	"heart_rate_min",
	"heart_rate_max",
	"heart_rate_range",
	"year",
	"season",
	"month",
	"day",
	"day_of_week",
	"temperature",
	"humidity",
]

df = df[newOrder]

df.head(2)

,startDate,endDate,distance_km,elevation_ascended_cm,duration,pace,speed,calories_active,calories_basal,calories_total,...,heart_rate_min,heart_rate_max,heart_rate_range,year,season,month,day,day_of_week,temperature,humidity
0,2024-03-09 17:03:18-03:00,2024-03-09 17:26:46-03:00,1.70326,1565.0,23.467609,13.778054,4.354751,83.9084,38.2454,122.1538,...,82,124,42,2024,Spring,3,9,5,6.19,5500.0
1,2024-03-16 14:36:43-03:00,2024-03-16 15:23:49-03:00,3.02969,5159.0,45.893084,15.147782,3.960976,187.3350,76.0570,263.3920,...,74,124,50,2024,Spring,3,16,5,8.69,5100.0


### Export clean csv

In [17]:
# Export to csv
df.to_csv("../data/processed/clean_walks.csv", index=False)